# Territorial analysis — ECB Rate Shock & SME Failures

## Objective
Test the equity dimension: did business failures accelerate more after the July 2022 ECB hike in economically fragile (ZFRR) departments? Failures are available by sector at national level *or* by department for all sectors combined, never crossed — so this runs as a separate department-level difference-in-differences, complementing the sectoral analysis.

**Note on the ZFRR coding.** The Observatoire des Territoires export uses an undocumented numeric code. It is mapped from the match between code counts and official figures: code 5 = FRR+ (4,456 communes vs ~4,500 published), code 4 = FRR socle (4+5 = 17,769 vs ~17,700), code 1 = transitional ex-ZRR (2,134 vs ~2,168). A department's treatment intensity is the share of its communes classified FRR (codes 4 or 5).

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning
warnings.simplefilter('ignore', ValueWarning)
from IPython.display import display

sys.path.insert(0, str(Path().resolve().parent))
from src.data_loader import load_failures_by_department, load_frr_communes
from src import cleaning

Path('../outputs').mkdir(exist_ok=True)
Path('../data/processed').mkdir(exist_ok=True)

fd = load_failures_by_department()
frr = load_frr_communes()
intensity = cleaning.zfrr_intensity_by_department(frr)
panel = cleaning.build_dept_panel(fd, intensity)
print('departments in panel:', panel['dept'].nunique())
print('FRR communes classified:', int(frr['classified'].sum()), '/', len(frr))

## 1. ZFRR intensity and treatment split
Each department is scored by the share of its communes classified FRR, then split at the median into treatment (more rural/fragile) and control.

In [ ]:
med = intensity.median()
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(intensity, bins=25, color='#3b6ea5')
ax.axvline(med, color='black', ls='--', lw=1.2, label=f'median = {med:.2f}')
ax.set_xlabel('Share of communes classified FRR (per department)')
ax.set_ylabel('Departments')
ax.set_title('ZFRR intensity across departments')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/zfrr_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

split = panel.groupby('treated')['dept'].nunique()
print('control departments:', int(split.get(0, 0)), '| treated departments:', int(split.get(1, 0)))

## 2. Parallel trends
Department-mean failures indexed to 2019, by group, with the excluded COVID/PGE window.

In [ ]:
pos = panel[panel['failures'] > 0].copy()
base2019 = pos[pos['date'].dt.year == 2019].groupby('dept')['failures'].mean()
pos['idx'] = pos['failures'] / pos['dept'].map(base2019) * 100
grp = (pos.groupby([pos['date'], 'treated'])['idx'].mean()
       .unstack().rename(columns={1: 'Treated (high ZFRR)', 0: 'Control (low ZFRR)'}))

fig, ax = plt.subplots(figsize=(12, 5))
grp.plot(ax=ax, color={'Treated (high ZFRR)': '#c0392b', 'Control (low ZFRR)': '#7f8c8d'}, lw=1.6)
ax.axvspan(cleaning.COVID_START, cleaning.COVID_END, alpha=0.12, color='gray',
           label='Excluded (COVID/PGE)')
ax.axvline(cleaning.ECB_FIRST_HIKE, color='black', ls='--', lw=1.2, label='July 2022 hike')
ax.set_ylabel('Failures, index 2019 = 100')
ax.set_title('Department failures by ZFRR group (mean index)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/dept_panel_windows.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Difference-in-differences
Department fixed effects, standard errors clustered by department. Three specifications: a binary median split, a continuous ZFRR-intensity interaction, and a baseline placebo.

In [ ]:
s = panel[panel['in_sample'] & (panel['failures'] > 0)].copy()
s['log_failures'] = np.log(s['failures'])

m = smf.ols('log_failures ~ C(dept) + post + did', data=s).fit(
    cov_type='cluster', cov_kwds={'groups': s['dept']})
print('--- Binary split (treated = above-median ZFRR) ---')
print(m.summary().tables[1].as_text())
print(f"DiD: {m.params['did']:+.4f} log ({np.expm1(m.params['did'])*100:+.1f}%), p={m.pvalues['did']:.3f}")

s['did_cont'] = s['post'] * s['zfrr_share']
mc = smf.ols('log_failures ~ C(dept) + post + did_cont', data=s).fit(
    cov_type='cluster', cov_kwds={'groups': s['dept']})
print(f"\nContinuous: did_cont={mc.params['did_cont']:+.4f}, p={mc.pvalues['did_cont']:.3f} (per +1.0 share)")

b = panel[panel['in_baseline'] & (panel['failures'] > 0)].copy()
b['log_failures'] = np.log(b['failures'])
b['fake_post'] = (b['date'] >= '2018-01-01').astype(int)
b['fake_did'] = b['fake_post'] * b['treated']
mp = smf.ols('log_failures ~ C(dept) + fake_post + fake_did', data=b).fit(
    cov_type='cluster', cov_kwds={'groups': b['dept']})
print(f"Placebo (fake 2018 break): {mp.params['fake_did']:+.4f}, p={mp.pvalues['fake_did']:.3f}")

**Results from the territorial DiD:** with ~96 departments the cluster-robust inference is far better powered than the nine-sector analysis. The binary and continuous specifications agree in sign; the baseline placebo is null, supporting the design.

## 4. DiD decomposition

In [ ]:
g = s.groupby(['treated', 'post'])['log_failures'].mean().unstack()
before_t, after_t = g.loc[1, 0], g.loc[1, 1]
before_c, after_c = g.loc[0, 0], g.loc[0, 1]
cf = before_t + (after_c - before_c)
fig, ax = plt.subplots(figsize=(7, 5))
x = [0, 1]
ax.plot(x, [before_t, after_t], 'o-', color='#c0392b', lw=2, label='Treated (high ZFRR)')
ax.plot(x, [before_c, after_c], 'o-', color='#7f8c8d', lw=2, label='Control (low ZFRR)')
ax.plot(x, [before_t, cf], 'o--', color='#c0392b', lw=1.5, alpha=0.5, label='Counterfactual')
ax.annotate('', xy=(1.04, after_t), xytext=(1.04, cf),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.4))
ax.text(1.06, (after_t + cf) / 2, f'DiD = {(after_t - cf):+.3f} log', va='center', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(['Baseline\n(2015-2019)', 'Post-hike\n(2022-07 on)'])
ax.set_ylabel('Mean log failures')
ax.set_title('Territorial DiD decomposition - high vs low ZFRR departments')
ax.legend(loc='upper left')
ax.set_xlim(-0.2, 1.4)
plt.tight_layout()
plt.savefig('../outputs/dept_did_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

panel.to_csv('../data/processed/dept_panel.csv', index=False)
intensity.to_frame().to_csv('../data/processed/zfrr_intensity.csv')
print('processed dept files written')

## Results summary

Contrary to the equity concern, failures did **not** accelerate more in fragile (ZFRR) departments after the rate hike — the point estimates are negative and significant, i.e. high-ZFRR departments saw a smaller post-2022 rise in failures than less-rural ones. This mirrors the sectoral result. A plausible reading: rural/fragile economies are less business-dense and more weighted toward activities (e.g. agriculture) that were heavily supported, so the post-pandemic failure rebound concentrated in more urban, dynamic areas.

**Caveats:**
1. Department failures are all-firms (no PME breakdown at this level), used as a close proxy since SMEs are ~99% of firms.
2. ZFRR intensity is the share of communes classified, unweighted by population or firm count (a refinement would weight by establishments).
3. The ZFRR numeric coding is inferred from official counts (see top note) and should be confirmed against the Observatoire des Territoires legend.